# FingerBall — Stage 2H Spin-off: 360° Stick Viewer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_Stage2H_Spinoff_360_Stick_Viewer.ipynb)

**データ提供者に渡すための簡易3Dスティック骨格ビューア**です。

- Stage 2Hで目視確認した**元2D座標を変更しません**。
- マウスドラッグで360°回転・ホイールで拡大縮小・フレームスライダーと再生ボタン。
- **骨格上の指名テキストは表示しません**。5指の色別凡例のみ。
- 遮蔽・不明点は欠損のまま表示せず、無理に埋めません。
- 奥行きは**未補正のMediaPipe相対Z**を参考利用（取得できなければZ=0の平面骨格）。したがって**実測3D・解剖学的に検証された3D再構成ではありません**。360°回転できても実際の奥行きが確定するわけではありません。
- **共有用の単独HTML**を生成します。閲覧者はColab/Pythonを必要としません。元動画・顔画像・患者IDなどをHTMLに埋め込みません。

**推奨**：先生へ送る前に188〜193フレームを目視確認してください。

## 0. セットアップとDrive接続

In [ ]:
%pip -q install "plotly>=5.18" "pandas>=2" "numpy>=1.24"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from IPython.display import display,HTML
import hashlib,json,zipfile
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

## 1. 入力と表示設定

通常はそのまま実行できます。Stage 2Hの最終版が別名・別フォルダの場合、`CORRECTED_CSV`を変更してください。Stage 2DのZが存在しない場合は平面スティックを生成します。

In [ ]:
PROJECT=Path('/content/drive/MyDrive/母指回内トラッキング')
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
CORRECTED_CSV=PROJECT/'FingerBall_v2_stage2h_results'/VIDEO_STEM/'hand_21_landmarks_2d_corrected.csv'
RAW_CSV=PROJECT/'FingerBall_v2_stage2d_results'/VIDEO_STEM/'hand_21_landmarks_long.csv'
OUTPUT_DIR=PROJECT/'FingerBall_v2_stage2h_results'/VIDEO_STEM/'spinoff_360_stick'
START_FRAME,END_FRAME=185,198
INITIAL_FRAME=190
THROWING_HAND='Left'     # 手動で確認済み
Z_SCALE=1.0             # MediaPipe相対Zの表示倍率のみ。正確な深度を意味しない
POINT_SIZE=4
LINE_WIDTH=7
SHOW_AXES=False
if not CORRECTED_CSV.is_file():
    raise FileNotFoundError('Stage 2H補正CSVを指定してください: '+str(CORRECTED_CSV))
if not (0<Z_SCALE<=20):raise ValueError('Z_SCALEは0より大きく20以下としてください')
if not (START_FRAME<=INITIAL_FRAME<=END_FRAME):raise ValueError('INITIAL_FRAMEが範囲外')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
frozen_hash=hashlib.sha256(CORRECTED_CSV.read_bytes()).hexdigest()
print('2D source:',CORRECTED_CSV)
print('2D source SHA256:',frozen_hash)
print('Raw Z reference:',RAW_CSV,'exists:',RAW_CSV.is_file())
print('Output:',OUTPUT_DIR)

## 2. 手動2D座標の読込とZの参照付与

修正済みXYは読取専用で保持します。Zは別ソースの表示情報として扱い、2D修正後にZまで正確になったとは解釈しません。

In [ ]:
corrected=pd.read_csv(CORRECTED_CSV)
required={'frame','landmark_id','x_px','y_px'}
if missing:=required-set(corrected.columns):
    raise ValueError('Stage 2H補正CSVに必要な列がありません: '+str(missing))
if corrected.duplicated(['frame','landmark_id']).any():
    raise ValueError('重複 frame/landmark_id')
frames=list(range(START_FRAME,END_FRAME+1))
src=corrected[corrected.frame.between(START_FRAME,END_FRAME)].copy()
for f in frames:
    ids=src.loc[src.frame==f,'landmark_id'].astype(int).tolist()
    if len(ids)!=21 or sorted(ids)!=list(range(21)):
        raise ValueError(f'Frame {f}: 21ランドマークが揃っていません')
if 'annotation_status' not in src.columns:
    src['annotation_status']='unknown'
if RAW_CSV.is_file():
    raw=pd.read_csv(RAW_CSV)
    if not {'frame','landmark_id','z_relative'}.issubset(raw.columns):
        raise ValueError('RAW_CSVにz_relative列がありません')
    if raw.duplicated(['frame','landmark_id']).any():raise ValueError('RAW_CSVに重複')
    zref=raw[['frame','landmark_id','z_relative']].rename(columns={'z_relative':'raw_z_reference'})
    src=src.merge(zref,on=['frame','landmark_id'],how='left',validate='one_to_one')
    z_origin='MediaPipe raw relative-Z (not manually corrected)'
else:
    src['raw_z_reference']=0.0
    z_origin='Z=0 flat skeleton: no reference depth'
print('XY state counts:',src.annotation_status.value_counts(dropna=False).to_dict())
print('Z mode:',z_origin)
# X right, Y up; normalize XY by original image width (1206 px)
IMAGE_WIDTH=1206.0
points_by_frame={}
missing_by_frame={}
for f in frames:
    a=src[src.frame==f].sort_values('landmark_id')
    xy=a[['x_px','y_px']].to_numpy(dtype=float)
    status=a.annotation_status.fillna('unknown').to_list()
    # honor explicitly occluded points even if stale numeric values remain
    for i,st in enumerate(status):
        if st in ('occluded','missing'):
            xy[i,:]=np.nan
    z=a.raw_z_reference.to_numpy(dtype=float)
    if RAW_CSV.is_file():
        # raw depth can be missing even when manually verified XY exists
        if np.isfinite(z[0]):z=z-z[0]
        else:z=np.full(21,np.nan)
    z=np.where(np.isfinite(z),z,0.0) # depth missing -> planar fallback at that landmark
    coords=np.column_stack([(xy[:,0]-xy[0,0])/IMAGE_WIDTH,
                            -(xy[:,1]-xy[0,1])/IMAGE_WIDTH,
                            -z*Z_SCALE])
    if not np.isfinite(xy[0]).all():
        # wrist missing: use center of available joints, still do not invent wrist point
        center=np.nanmean(xy,axis=0)
        coords[:,:2]=np.column_stack([(xy[:,0]-center[0])/IMAGE_WIDTH,
                                     -(xy[:,1]-center[1])/IMAGE_WIDTH])
    invalid=~np.isfinite(xy).all(axis=1)
    coords[invalid,:]=np.nan
    points_by_frame[f]=coords
    missing_by_frame[f]=int(invalid.sum())
print('Missing / occluded points per frame:',missing_by_frame)

## 3. 360°回転できるスティックビューア

**自由回転**：マウスドラッグ。**ズーム**：ホイール。**フレーム移動**：下のスライダー、またはPlay/Pause。5指の凡例のみ表示し、関節名は骨格上に一切記載しません。

In [ ]:
FINGER_PATHS={
    'Thumb':[0,1,2,3,4],
    'Index':[0,5,6,7,8],
    'Middle':[0,9,10,11,12],
    'Ring':[0,13,14,15,16],
    'Little':[0,17,18,19,20]}
FINGER_COLORS={'Thumb':'#f59e0b','Index':'#06b6d4','Middle':'#d946ef',
               'Ring':'#cbd5e1','Little':'#a3a3a3'}
def traces(f):
    pts=points_by_frame[f]
    result=[]
    for name,ids in FINGER_PATHS.items():
        X=[];Y=[];Z=[]
        for a,b in zip(ids[:-1],ids[1:]):
            v,w=pts[a],pts[b]
            if np.isfinite(v).all() and np.isfinite(w).all():
                X.extend([v[0],w[0],None])
                Y.extend([v[1],w[1],None])
                Z.extend([v[2],w[2],None])
        result.append(go.Scatter3d(
            x=X,y=Y,z=Z,mode='lines',name=name,showlegend=True,
            line=dict(color=FINGER_COLORS[name],width=LINE_WIDTH),
            hoverinfo='name'))
    result.append(go.Scatter3d(
        x=pts[:,0],y=pts[:,1],z=pts[:,2],
        mode='markers',name='joints',showlegend=False,
        hoverinfo='skip',marker=dict(color='#f8fafc',size=POINT_SIZE)))
    return result
valid_points=np.vstack([p[np.isfinite(p).all(axis=1)] for p in points_by_frame.values()
                        if np.isfinite(p).all(axis=1).any()])
if len(valid_points)==0:raise ValueError('表示できる点がありません')
# Fixed axis ranges for all frames prevent apparent size changes
limits=[]
for ax in range(3):
    lim=float(np.nanmax(np.abs(valid_points[:,ax])))
    limits.append(max(.04,lim*1.28))
scene_axis=lambda name,lim:dict(title=name,range=[-lim,lim],
       showgrid=SHOW_AXES,showticklabels=SHOW_AXES,zeroline=False,
       showbackground=False,visible=True)
viewer=go.Figure(
    data=traces(INITIAL_FRAME),
    frames=[go.Frame(name=str(f),data=traces(f)) for f in frames])
viewer.update_layout(
    template='plotly_dark',height=720,
    title=dict(text='FingerBall | Stage 2H reviewed hand skeleton | pseudo-3D',x=.5),
    margin=dict(l=0,r=0,t=72,b=30),
    legend=dict(orientation='h',y=1.06,x=.5,xanchor='center',font=dict(size=12)),
    scene=dict(
        xaxis=scene_axis('X',limits[0]),
        yaxis=scene_axis('Y',limits[1]),
        zaxis=scene_axis('Relative Z',limits[2]),
        aspectmode='cube',
        dragmode='orbit',
        uirevision='preserve-rotation',
        camera=dict(eye=dict(x=1.3,y=1.5,z=1.2))),
    sliders=[dict(active=frames.index(INITIAL_FRAME),
        currentvalue=dict(prefix='Frame: ',font=dict(size=13)),
        x=.12,len=.76,y=0.02,steps=[
        dict(label=str(f),method='animate',
             args=[[str(f)],dict(mode='immediate',
                frame=dict(duration=0,redraw=True),transition=dict(duration=0))])
        for f in frames])],
    updatemenus=[dict(type='buttons',x=.0,y=0.01,direction='left',showactive=False,
        buttons=[
          dict(label='▶ Play',method='animate',args=[
               None,dict(frame=dict(duration=350,redraw=True),
                        transition=dict(duration=0),fromcurrent=True,mode='immediate')]),
          dict(label='Ⅱ Pause',method='animate',args=[
               [None],dict(mode='immediate',frame=dict(duration=0,redraw=True),
                           transition=dict(duration=0))])])],
    annotations=[dict(text='Rotation is illustrative: 2D manually reviewed; depth is unvalidated.',
                      showarrow=False,x=.5,y=-.04,xref='paper',yref='paper',
                      font=dict(size=11,color='#aab5c3'))])
viewer.show(config=dict(displaylogo=False,scrollZoom=True,responsive=True))

## 4. データ提供者向け単独HTMLの出力（常時表示の再生操作バー）

HTMLの**骨格画面の上部**に、フレームスライダー・現在のフレーム番号・▶ Play・Ⅱ Pauseを配置します。Plotlyの内部メニューが画面外へ隠れることがあっても、外側の操作バーは常時表示されます。マウスで自由回転・拡大できます。

In [ ]:
html_path=OUTPUT_DIR/'FingerBall_Stage2H_360deg_stick_viewer.html'

# Plotly data/frames and plotly.js are embedded locally: the recipient
# needs only a browser. Custom controls live OUTSIDE the 3D canvas.
# Do not rely on Plotly's internal slider position (can be clipped).
import html as html_std
from IPython.display import IFrame

plot_for_share=go.Figure(viewer)
plot_for_share.update_layout(
    sliders=[],updatemenus=[],
    height=680,
    scene=dict(uirevision='preserve-rotation'),
    margin=dict(l=0,r=0,t=75,b=20))
plot_html=pio.to_html(
    plot_for_share,full_html=False,include_plotlyjs=True,
    div_id='fingerball_spin_plot',
    config={'displaylogo':False,'scrollZoom':True,'responsive':True})

frame_numbers=[int(f) for f in frames]
frame_values=json.dumps(frame_numbers)
intro=(
    f"Stage 2H manual-reviewed XY | {THROWING_HAND} hand | "
    f"relative depth: {z_origin}. "
    "Pseudo-3D only; not calibrated real 3D.")
template=r"""<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>FingerBall — 360° Stick Viewer</title>
<style>
html,body{margin:0;background:#0b1220;color:#e2e8f0;font-family:system-ui,-apple-system,"Segoe UI",sans-serif}
.wrap{max-width:1200px;margin:0 auto;padding:14px 16px}
h1{font-size:20px;margin:6px 0 8px}
.note{font-size:12px;color:#cbd5e1;line-height:1.5;margin:6px 0 10px}
.toolbar{position:relative;z-index:5;background:#1e293b;border:1px solid #475569;
border-radius:10px;padding:14px;margin:8px 0 12px;
display:flex;flex-wrap:wrap;align-items:center;gap:12px}
.toolbar button{border:1px solid #94a3b8;border-radius:7px;background:#334155;
color:#fff;font-size:15px;font-weight:600;cursor:pointer;padding:9px 16px;min-width:92px}
.toolbar button:hover{background:#475569}
.toolbar button:focus-visible,.toolbar input:focus-visible{outline:3px solid #38bdf8}
.toolbar input[type=range]{flex:1 1 240px;min-width:125px;accent-color:#38bdf8;cursor:pointer}
.frameNumber{white-space:nowrap;font-weight:700;font-variant-numeric:tabular-nums}
.plotFrame{border:1px solid #334155;border-radius:10px;overflow:hidden;background:#111827}
#fingerball_spin_plot{width:100%;min-height:640px}
#fbStatus{min-height:16px;color:#cbd5e1;font-size:12px}
</style></head><body>
<div class="wrap">
<h1>FingerBall — 360° Stick Viewer</h1>
<div class="note">__INTRO__</div>
<div class="toolbar" aria-label="フレーム再生コントロール">
 <button id="fbPlay" type="button">▶ Play</button>
 <button id="fbPause" type="button">Ⅱ Pause</button>
 <label class="frameNumber" for="fbFrame">Frame: <span id="fbCurrent">__FIRST__</span></label>
 <input id="fbFrame" aria-label="時間軸フレームスライダー" type="range"
        min="0" max="__LAST_INDEX__" step="1" value="__INITIAL_INDEX__">
 <span id="fbCounter" class="frameNumber"></span>
</div>
<div class="plotFrame">__PLOT__</div>
<p id="fbStatus">マウスで360°回転、ホイールで拡大・縮小。上の操作バーでフレームを選択・再生できます。</p>
</div>
<script>
(function(){
'use strict';
const frames=__FRAME_VALUES__;
const plot=document.getElementById('fingerball_spin_plot');
const bar=document.getElementById('fbFrame');
const displayFrame=document.getElementById('fbCurrent');
const counter=document.getElementById('fbCounter');
const play=document.getElementById('fbPlay');
const pause=document.getElementById('fbPause');
const status=document.getElementById('fbStatus');
let index=__INITIAL_INDEX__, timer=null;
const intervalMs=350;
function render(i){
 index=Math.max(0,Math.min(frames.length-1,Math.round(Number(i))));
 bar.value=String(index);
 displayFrame.textContent=String(frames[index]);
 counter.textContent=(index+1)+' / '+frames.length;
 if(typeof Plotly!=='undefined' && plot && plot.data){
  Plotly.animate(plot,[String(frames[index])],{
    mode:'immediate',
    frame:{duration:0,redraw:true},
    transition:{duration:0}
  }).catch(function(error){status.textContent='描画エラー: '+error;});
 }else{
  status.textContent='3D描画の読み込みを確認してください。';
 }
}
function stop(){if(timer!==null){clearInterval(timer);timer=null;}play.disabled=false;}
function start(){
 if(timer!==null)return;
 if(index>=frames.length-1)index=-1;
 play.disabled=true;
 render(index+1);
 timer=setInterval(function(){
  if(index>=frames.length-1){stop();return;}
  render(index+1);
 },intervalMs);
}
bar.addEventListener('input',function(){stop();render(this.value);});
play.addEventListener('click',start);
pause.addEventListener('click',stop);
render(index);
})();
</script>
</body></html>"""
template=template.replace('__PLOT__',plot_html)
template=template.replace('__INTRO__',html_std.escape(intro))
template=template.replace('__FRAME_VALUES__',frame_values)
template=template.replace('__FIRST__',str(INITIAL_FRAME))
template=template.replace('__INITIAL_INDEX__',str(frames.index(INITIAL_FRAME)))
template=template.replace('__LAST_INDEX__',str(len(frames)-1))
html_path.write_text(template,encoding='utf-8')

# Guard against accidental export without the requested controls.
exported=html_path.read_text(encoding='utf-8')
for must_have in ('id="fbFrame"','id="fbPlay"','id="fbPause"',
                  'id="fbCurrent"','Plotly.animate','fingerball_spin_plot'):
    if must_have not in exported:
        raise AssertionError('HTML control missing: '+must_have)

readme=OUTPUT_DIR/'README_share_with_provider.txt'
readme.write_text(
    'FingerBall Stage 2H | 360-degree stick viewer\n'
    'HTMLをブラウザで開くと、画面上部にフレームスライダーとPlay/Pauseボタンが表示されます。\n'
    'マウスで骨格を回転し、ホイールで拡大・縮小できます。\n'
    f'対象フレーム: {START_FRAME}〜{END_FRAME}、投球手: {THROWING_HAND}（手動指定）\n'
    '手動アノテーション済みの2D位置を保持しています。\n'
    f'奥行き: {z_origin}。実測3Dではありません。\n'
    '遮蔽点は表示せず、近傍点の接続も行いません。\n'
    '撮影許諾・研究倫理・共有範囲をご確認ください。\n',
    encoding='utf-8')

manifest={'stage':'2H_spinoff_360_controls_v2',
    'original_corrected_csv':str(CORRECTED_CSV),
    'original_sha256_before':frozen_hash,
    'original_sha256_after':hashlib.sha256(CORRECTED_CSV.read_bytes()).hexdigest(),
    'depth_reference':z_origin,'frames':[START_FRAME,END_FRAME],
    'manual_hand':THROWING_HAND,'z_scale_display_only':Z_SCALE,
    'controls':'standalone HTML external range slider + Play/Pause'}
if manifest['original_sha256_before']!=manifest['original_sha256_after']:
    raise RuntimeError('入力CSVが実行中に変更されました')
(OUTPUT_DIR/'viewer_manifest.json').write_text(
    json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')

zip_path=OUTPUT_DIR/'FingerBall_Stage2H_360deg_stick_for_provider.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    archive.write(html_path,arcname=html_path.name)
    archive.write(readme,arcname=readme.name)
print('HTML:',html_path,round(html_path.stat().st_size/1e6,2),'MB')
print('ZIP:',zip_path,round(zip_path.stat().st_size/1e6,2),'MB')
print('Verified: slider, Play and Pause controls included in standalone HTML')
display(HTML('<p><b>完了：</b>生成したHTMLをダウンロードしてブラウザで開いてください。操作バーは骨格の上に表示されます。</p>'))